# Sheet 4.1: Doing and writing research

**Roadmap topic 4.1** (Fanout 09.01). Read first: Schulman's *Opinionated Guide to ML Research*, Keshav's *How to Read a Paper*, Peyton Jones's *How to Write a Great Research Paper*. Look at past ML Reproducibility Challenge reports for the format.

This sheet has two parts: (A) the statistics and engineering that make results trustworthy, with auto-checks, and (B) the project itself, run as a sequence of deliverables. Part B is your **Step 4 research artifact**.

**Rules:** fill every `# YOUR CODE` cell, never edit `CHECK` cells. Part A runs on CPU.

In [ ]:
import math, random, os, numpy as np, torch
from scipy import stats

# Part A: making results trustworthy

## Exercise 1: Determinism

`seed_everything(seed)` seeds Python, NumPy and PyTorch (CPU and CUDA), sets `torch.backends.cudnn.deterministic = True`, `benchmark = False`, and calls `torch.use_deterministic_algorithms(True, warn_only=True)`. Also set `CUBLAS_WORKSPACE_CONFIG=:4096:8`, which deterministic cuBLAS needs on GPU.

`run_experiment(seed)` (given below) must return bit-identical results for the same seed.

In [ ]:
def seed_everything(seed):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
def run_experiment(seed):
    seed_everything(seed)
    X = torch.randn(256, 20); y = (X[:, 0] + 0.5 * np.random.randn(256).astype(np.float32) > 0).long()
    m = torch.nn.Sequential(torch.nn.Linear(20, 32), torch.nn.Dropout(0.2), torch.nn.ReLU(), torch.nn.Linear(32, 2))
    opt = torch.optim.Adam(m.parameters(), 1e-2); idx = list(range(256)); random.shuffle(idx)
    for i in range(0, 256, 32):
        b = idx[i:i + 32]; opt.zero_grad(); torch.nn.functional.cross_entropy(m(X[b]), y[b]).backward(); opt.step()
    return torch.cat([p.detach().flatten() for p in m.parameters()])

In [ ]:
# ---- CHECK 1 (do not edit) ----
def _check1():
    a, b, c = run_experiment(3), run_experiment(3), run_experiment(4)
    assert torch.equal(a, b), "same seed must give bit-identical weights"
    assert not torch.equal(a, c)
    assert os.environ.get("CUBLAS_WORKSPACE_CONFIG") == ":4096:8"
    print("PASS")
_check1()

## Exercise 2: How many seeds do you need?

You expect your method to improve accuracy by `delta` and seed-to-seed std is `sigma` for both methods. For a two-sided Welch t-test at level `alpha`, find the smallest number of seeds per method that achieves `power`.

- `power_two_sample(n, delta, sigma, alpha)`: power using the **noncentral t** distribution (`scipy.stats.nct`) with `df = 2n - 2` and noncentrality `delta / (sigma·√(2/n))`.
- `seeds_needed(delta, sigma, alpha=0.05, power=0.8)`.

In [ ]:
def power_two_sample(n, delta, sigma, alpha=0.05):
    # YOUR CODE
    raise NotImplementedError
def seeds_needed(delta, sigma, alpha=0.05, power=0.8):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 2 (do not edit) ----
def _check2():
    assert seeds_needed(1.0, 1.0) == 17, seeds_needed(1.0, 1.0)
    assert seeds_needed(0.5, 0.3) == 7, seeds_needed(0.5, 0.3)
    r = np.random.default_rng(0); n = seeds_needed(0.5, 0.3); hits = 0
    for _ in range(4000):
        a = r.normal(0, .3, n); b = r.normal(.5, .3, n); hits += stats.ttest_ind(a, b, equal_var=False).pvalue < 0.05
    assert abs(hits / 4000 - power_two_sample(n, .5, .3)) < 0.04, "simulated power disagrees with your formula"
    print(f"PASS  0.5-point gain with 0.3 std needs {n} seeds per method; a 1-sigma effect needs {seeds_needed(1,1)}")
_check2()

**Written (2a):** Papers often report "our method beats the baseline" from 3 seeds. With your function, what effect size (in units of seed std) can 3 seeds detect with 80% power? What does that imply about the many small-gain results you read?

**Your answer:**

## Exercise 3: Many comparisons

You compare your method against the baseline on 8 benchmarks and get 8 p-values. Implement Holm-Bonferroni: `holm(pvals, alpha)` returns a boolean list of which hypotheses are rejected while controlling the family-wise error rate at `alpha`.

In [ ]:
def holm(pvals, alpha=0.05):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 3 (do not edit) ----
def _check3():
    p = [0.010, 0.04, 0.03, 0.005, 0.2, 0.0001, 0.011, 0.6]
    assert holm(p) == [False, False, False, True, False, True, False, False], holm(p)
    assert holm([0.01, 0.02], 0.05) == [True, True] and holm([0.03, 0.04], 0.05) == [False, False]
    print("PASS  only 2 of 8 benchmarks survive the correction (naive p<0.05 would claim 6)")
_check3()

## Exercise 4: The results table

`results_table(runs)` takes `{method: {benchmark: [score per seed]}}` and returns a Markdown table with one row per method, `mean ± 95% CI half-width` (t-distribution) per benchmark to 1 decimal, and the best mean per column in **bold**. This is the table that goes into your paper; generate it from code so it can never drift from the logs.

In [ ]:
def results_table(runs):
    # YOUR CODE
    raise NotImplementedError

In [ ]:
# ---- CHECK 4 (do not edit) ----
def _check4():
    runs = {"baseline": {"GSM8K": [40.1, 41.3, 39.8], "MATH": [12.0, 12.5, 11.0]},
            "ours": {"GSM8K": [43.0, 44.2, 42.9], "MATH": [11.2, 11.6, 11.9]}}
    t = results_table(runs); lines = t.strip().splitlines()
    assert lines[0].replace(" ", "") == "|Method|GSM8K|MATH|" and len(lines) == 4
    h = stats.t.ppf(.975, 2) * np.std([43.0, 44.2, 42.9], ddof=1) / math.sqrt(3)
    assert f"**43.4 ± {h:.1f}**" in lines[3], lines[3]
    assert "**11.8" in lines[2] and "**" not in lines[3].split("|")[3], "bold must mark the best mean per column"
    print("PASS\n" + t)
_check4()

# Part B: your research project (2-4 weeks)

Work through these deliverables in order. Each ends with something you can show.

### B1. Choose a question (2-3 days)
Read 10 abstracts and 3 full papers from 2024-2026 in an area you've touched in Steps 2-3 (GRPO variants, LoRA behaviour, normalisation, small-model reasoning, tokenisation). Pick **one claim** that (a) can be tested at ≤ 1B parameters, (b) needs ≤ 40 T4-hours, (c) would be interesting whether it holds or not.

**Deliverable:** a one-page proposal in this cell: claim, why it matters, exact experiment, baselines, metrics, compute estimate (T4-hours, from your sheet 2.2 MFU numbers), and the result that would falsify it.

### B2. Pre-register (half a day)
Before running anything, write down hypotheses, metrics, number of seeds (use Exercise 2), and the comparisons you'll correct for (Exercise 3). Commit this to your repo with a timestamp.

### B3. Reproduce the baseline first (3-5 days)
Get the paper's baseline number (or the closest small-scale analogue) within its reported variance. If you can't, that **is** a result: document every deviation.

### B4. Run the experiment (1-2 weeks)
Log every run (sheet 4.2's tracker), including failures. Keep a dated lab notebook in the repo: what you ran, why, what happened.

### B5. Write it (1 week)
NeurIPS workshop template on Overleaf, 4-8 pages: abstract, introduction (the question and why), method, results with Exercise 4 tables and error bars, limitations, reproducibility statement. Every figure regenerates from one command. Ask someone outside ML to read the introduction; fix whatever confuses them.

### B6. Publish
Post to arXiv (first submissions may need an endorsement from an existing author in the category; ask a researcher whose work you build on) or submit to a workshop or the ML Reproducibility Challenge. Write a short thread/blog post with the main figure.

## Checkpoint
Every number and figure in your paper can be regenerated from the repo with a single command, by someone who has never seen it.

---
### Hints for self-grading
- 2a: with n = 3, about 3.1σ (check with `power_two_sample`). Gains smaller than ~3 seed-stds from 3-seed comparisons are mostly noise.